# Honest validation of biofilter models — full analysis (both datasets)

**How to run on Kaggle**
1. Add two datasets as inputs: `toluene_cofeed_clean.csv` (Dataset A) and `DataSet.xlsx` (Dataset B, Zarei et al. 2025).
2. Menu **Run → Run all**. CPU is enough. Total run time: a few minutes (the adaptive-model forecasts are the slowest part).
3. All tables (`.csv`) and figures (`.png`) are saved in `/kaggle/working/results`.

**Datasets**
- **A** — Xue et al. 2024, PLOS One, doi:10.1371/journal.pone.0302487. Two biotrickling filters, toluene with ethyl acetate (reactor A) or n-hexane (reactor B), 4 phases, 72 samples.
- **B** — Zarei et al. 2025, PLOS One, doi:10.1371/journal.pone.0339352; data doi:10.6084/m9.figshare.30266404. Biochar biofilter for H₂S, 18 conditions × 3 repeats = 54 rows.


Run everything with **Run → Run all** (about 15–20 min on a standard CPU). Inputs: `toluene_cofeed_clean.csv` and `DataSet.xlsx` (see README).

In [ ]:
# ============ 0. Setup: imports, paths, shared helpers ============
import glob, os, warnings
import numpy as np, pandas as pd
from scipy.optimize import least_squares
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import KFold, ShuffleSplit, LeaveOneGroupOut
from xgboost import XGBRegressor
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

OUT = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
os.makedirs(OUT, exist_ok=True)

def find(name):
    hits = (glob.glob(f"/kaggle/input/**/{name}", recursive=True) or glob.glob(name)
            or glob.glob(f"data/{name}") or glob.glob(f"**/{name}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{name} not found: add it as a Kaggle input (Add Input -> Your Datasets)")
    return hits[0]

plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
C1, C2, C3, C4, C5 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
INK, INK2, GRID, SEP = "#0b0b0b", "#52514e", "#ecebe6", "#d0cfca"

def mae(e):  return float(np.mean(np.abs(e)))
def rmse(e): return float(np.sqrt(np.mean(np.square(e))))
def r2(yt, yp): return float(1 - np.sum((yp - yt) ** 2) / np.sum((yt - np.mean(yt)) ** 2))

def block_boot_diff(err_a, err_b, blocks, metric="mae", n=5000, seed=0):
    """metric(err_a) - metric(err_b) with a 95% CI, resampling whole blocks (conditions / phases)."""
    rng = np.random.default_rng(seed)
    ub = np.unique(blocks)
    idx = {b: np.where(blocks == b)[0] for b in ub}
    f = mae if metric == "mae" else rmse
    point = f(err_a) - f(err_b)
    boots = []
    for _ in range(n):
        ii = np.concatenate([idx[b] for b in rng.choice(ub, size=len(ub), replace=True)])
        boots.append(f(err_a[ii]) - f(err_b[ii]))
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return point, lo, hi

def verdict(lo, hi, worse="worse", better="better"):
    return worse if lo > 0 else better if hi < 0 else "no clear difference"

print("Results folder:", OUT)

## Part A — Dataset A: toluene with ethyl acetate / n-hexane (time-series leakage)

In [ ]:
# ============ A1. Data ============
df = pd.read_csv(find("toluene_cofeed_clean.csv"))
df["cofeed_in"] = pd.to_numeric(df["cofeed_in"], errors="coerce").fillna(0.0)
df = df.sort_values(["reactor", "day"]).reset_index(drop=True)
df["is_EA"] = ((df.reactor == "A") & (df.cofeed_in > 0)).astype(int)
df["is_hexane"] = ((df.reactor == "B") & (df.cofeed_in > 0)).astype(int)
df["day_in_phase"] = df.day - df.groupby(["reactor", "phase"]).day.transform("min")
df["group"] = df.reactor + "-" + df.phase
g = df.groupby("reactor")
df["y_prev"] = g.toluene_out.shift(1)          # last measured outlet (for forecasting only)
df["cofeed_prev"] = g.cofeed_in.shift(1)
df["phase_start"] = (df.phase != g.phase.shift(1)).astype(int)
y = df.toluene_out.values
TAU = 30.0                                       # EBRT, s
NAME = {"A": "Reactor A (+ ethyl acetate)", "B": "Reactor B (+ n-hexane)"}
print(len(df), "rows |", df.group.nunique(), "(reactor, phase) blocks")

# ---- mechanistic model: plug flow, first-order uptake, competitive inhibition by the co-pollutant
def mech_out(k0, ki, cin, cco):
    return cin * np.exp(-k0 * TAU / (1.0 + cco / ki))

def fit_mech(tr, reactor):
    """k0 from phase I rows, Ki from co-pollutant rows; uses only the rows given (no test data)."""
    same = tr[tr.reactor == reactor]
    p1 = same[same.phase == "I"]
    if len(p1) == 0:
        p1 = tr[tr.phase == "I"]
    k0 = float(np.mean(-np.log(p1.toluene_out / p1.toluene_in) / TAU))
    co = same[same.cofeed_in > 0]
    if len(co) == 0:
        co = tr[tr.cofeed_in > 0]
    if len(co) == 0:
        return k0, np.inf
    res_ = lambda l: (np.log(mech_out(k0, np.exp(l[0]), co.toluene_in.values, co.cofeed_in.values))
                      - np.log(co.toluene_out.values))
    sol = least_squares(res_, x0=[np.log(1000.0)], bounds=([np.log(10.0)], [np.log(1e6)]))
    return k0, float(np.exp(sol.x[0]))

def predict_mech(tr, te):
    out = np.empty(len(te))
    for r in te.reactor.unique():
        m = (te.reactor == r).values
        k0, ki = fit_mech(tr, r)
        out[m] = mech_out(k0, ki, te.toluene_in.values[m], te.cofeed_in.values[m])
    return out

In [ ]:
# ============ A2. Mechanistic model fitted on all data (diagnostic: where does the physics fail?) ============
rows, parts = [], []
for R, d in df.groupby("reactor"):
    k0, ki = fit_mech(d, R)
    d = d.copy()
    d["C_mech"] = mech_out(k0, ki, d.toluene_in.values, d.cofeed_in.values)
    d["resid"] = d.toluene_out - d.C_mech
    parts.append(d)
    print(f"Reactor {R}: k0 = {k0:.4f} 1/s, Ki = {ki:.0f} mg/m3")
    for P, gp in d.groupby("phase"):
        rows.append(dict(reactor=R, phase=P,
                         RE_obs=round(100 * (1 - gp.toluene_out / gp.toluene_in).mean(), 1),
                         RE_mech=round(100 * (1 - gp.C_mech / gp.toluene_in).mean(), 1),
                         bias_mg=round(gp.resid.mean(), 1), RMSE_mg=round(rmse(gp.resid), 1)))
mech_all = pd.concat(parts)
tabA2 = pd.DataFrame(rows)
print(tabA2.to_string(index=False))
tabA2.to_csv(f"{OUT}/A2_mechanistic_by_phase.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, (R, d) in zip(axes, mech_all.groupby("reactor")):
    for x in [62.5, 107.5, 152.5]:
        ax.axvline(x, color=SEP, lw=1, zorder=0)
    for x, p in zip([40, 85, 130, 175], ["I", "II", "III", "IV"]):
        ax.text(x, 0.97, p, ha="center", va="top", color=INK2, transform=ax.get_xaxis_transform())
    ax.plot(d.day, d.toluene_out, "o", ms=5, color=C1 if R == "A" else C2, label="Observed")
    ax.plot(d.day, d.C_mech, "-", lw=2, color=INK2, label="Mechanistic")
    ax.set_title(NAME[R], loc="left", pad=12); ax.set_xlabel("Day"); ax.grid(axis="y", color=GRID, lw=0.8)
axes[0].set_ylabel("Toluene outlet (mg/m³)"); axes[0].set_ylim(-30, 1250)
axes[1].legend(frameon=False, loc="lower right")
fig.tight_layout(); fig.savefig(f"{OUT}/Fig2_A_mechanistic_trajectories.png", dpi=200); plt.show()

In [ ]:
# ============ A3. Direct ML vs baselines under three ways of testing ============
FEATURES = ["toluene_in", "cofeed_in", "is_EA", "is_hexane", "day_in_phase"]

def scaled(model):
    return TransformedTargetRegressor(regressor=make_pipeline(StandardScaler(), model), transformer=StandardScaler())

ML_A = {
    "Ridge": lambda: scaled(Ridge(alpha=1.0)),
    "SVR-RBF": lambda: scaled(SVR(C=3.0, epsilon=0.05, gamma="scale")),
    "Random Forest": lambda: RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=0),
    "XGBoost": lambda: XGBRegressor(n_estimators=300, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=0),
}
MODELS_A = ["Train mean", "Mechanistic"] + list(ML_A)

def run_fold(tr, te):
    p = {"Train mean": np.full(len(te), tr.toluene_out.mean()), "Mechanistic": predict_mech(tr, te)}
    for name, make in ML_A.items():
        m = make(); m.fit(tr[FEATURES], tr.toluene_out)
        p[name] = m.predict(te[FEATURES])
    return p

def evaluate(folds):
    pred = {}
    for tr_idx, te_idx in folds:
        p = run_fold(df.iloc[tr_idx], df.iloc[te_idx])
        for k, v in p.items():
            pred.setdefault(k, np.full(len(df), np.nan))[te_idx] = v
    return pred

def metrics_a(p, mask=None):
    m = ~np.isnan(p) if mask is None else mask & ~np.isnan(p)
    e = p[m] - y[m]
    return dict(MAE=mae(e), RMSE=rmse(e), Bias=float(np.mean(e)), R2=r2(y[m], p[m]))

rows, pred_store = [], {}
rand = [evaluate([(tr, te) for tr, te in KFold(5, shuffle=True, random_state=s).split(df)]) for s in range(5)]
for name in MODELS_A:
    ms = pd.DataFrame([metrics_a(r[name]) for r in rand]).mean()
    rows.append(dict(scheme="Random 5-fold (as in literature)", model=name, **ms))
SCHEMES_A = {
    "Leave-one-phase-out": [(np.where(df.group != g_)[0], np.where(df.group == g_)[0]) for g_ in sorted(df.group.unique())],
    "Cross-reactor": [(np.where(df.reactor != r_)[0], np.where(df.reactor == r_)[0]) for r_ in ["A", "B"]],
}
for scheme, folds in SCHEMES_A.items():
    pred = evaluate(folds); pred_store[scheme] = pred
    for name in MODELS_A:
        rows.append(dict(scheme=scheme, model=name, **metrics_a(pred[name])))
res_a = pd.DataFrame(rows)
res_a[["MAE", "RMSE", "Bias"]] = res_a[["MAE", "RMSE", "Bias"]].round(0); res_a["R2"] = res_a["R2"].round(2)
for s in res_a.scheme.unique():
    print(f"\n=== {s}  (toluene outlet, mg/m3)")
    print(res_a[res_a.scheme == s].drop(columns="scheme").to_string(index=False))
res_a.to_csv(f"{OUT}/A3_validation_results.csv", index=False)

lp = pred_store["Leave-one-phase-out"]
per_phase = pd.DataFrame([{"held-out": g_, **{k: round(metrics_a(v, (df.group == g_).values)["MAE"]) for k, v in lp.items()}}
                          for g_ in sorted(df.group.unique())])
print("\n=== MAE per held-out phase (leave-one-phase-out)")
print(per_phase.to_string(index=False))
per_phase.to_csv(f"{OUT}/A3_lopo_per_phase.csv", index=False)

# ---- statistics: differences vs references, 95% CI by bootstrapping (reactor, phase) blocks
rows = []
for scheme in SCHEMES_A:
    p = pred_store[scheme]
    for ref in ["Train mean", "Mechanistic"]:
        for k in MODELS_A:
            if k == ref:
                continue
            d_, lo, hi = block_boot_diff(p[k] - y, p[ref] - y, df.group.values, "mae")
            rows.append(dict(scheme=scheme, model=k, reference=ref, dMAE=round(d_), CI95_low=round(lo),
                             CI95_high=round(hi), verdict=verdict(lo, hi)))
stats_a = pd.DataFrame(rows)
print("\n=== MAE difference vs reference (positive = worse than reference), 95% CI")
print(stats_a.to_string(index=False))
stats_a.to_csv(f"{OUT}/A3_stats.csv", index=False)

In [ ]:
# ============ A4. Forecasting 5 days ahead, strictly forward in time ============
FIRST_FORECAST_DAY = 40
HYB_FEATS = ["r_prev", "cofeed_in", "cofeed_prev", "phase_start", "day_in_phase", "is_EA", "is_hexane"]
DIR_FEATS = ["y_prev", "toluene_in", "cofeed_in", "cofeed_prev", "phase_start", "day_in_phase", "is_EA", "is_hexane"]
def ridge(): return make_pipeline(StandardScaler(), Ridge(alpha=1.0))
def xgb():   return XGBRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, subsample=0.8, random_state=0)

def add_mech(frame, params):
    c = np.empty(len(frame))
    for r_, (k0, ki) in params.items():
        m = (frame.reactor == r_).values
        c[m] = mech_out(k0, ki, frame.toluene_in.values[m], frame.cofeed_in.values[m])
    return c

records = []
for d0 in sorted(df.day.unique()):
    if d0 < FIRST_FORECAST_DAY:
        continue
    tr = df[df.day < d0].copy(); te = df[df.day == d0].copy()
    params = {r_: fit_mech(tr, r_) for r_ in ["A", "B"]}
    tr["C_mech"] = add_mech(tr, params); te["C_mech"] = add_mech(te, params)
    tr["r"] = tr.toluene_out - tr.C_mech
    tr["r_prev"] = tr.groupby("reactor").r.shift(1)
    last = tr.sort_values("day").groupby("reactor").tail(1).set_index("reactor")
    te["r_prev"] = te.reactor.map(last.r)
    trh = tr.dropna(subset=["r_prev"]); trd = tr.dropna(subset=["y_prev"])
    pred = {"Persistence": te.y_prev.values, "Mechanistic": te.C_mech.values,
            "Hybrid: mech + last residual": te.C_mech.values + te.r_prev.values}
    for name, make in [("Ridge", ridge), ("XGBoost", xgb)]:
        pred[f"Hybrid: mech + {name} residual"] = te.C_mech.values + make().fit(trh[HYB_FEATS], trh.r).predict(te[HYB_FEATS])
        pred[f"Direct {name} (with lag)"] = make().fit(trd[DIR_FEATS], trd.toluene_out).predict(te[DIR_FEATS])
    for i, (_, row) in enumerate(te.iterrows()):
        for k, v in pred.items():
            records.append(dict(reactor=row.reactor, day=row.day, phase=row.phase, phase_start=int(row.phase_start),
                                observed=row.toluene_out, model=k, pred=float(v[i])))
fc = pd.DataFrame(records); fc["err"] = fc.pred - fc.observed

In [ ]:
# ============ A5. Adaptive mechanistic model (microbial activity with memory) ============
S_CO = 1000.0
LB = np.array([np.log(0.005), 0.0, 0.0, -0.9, np.log(1.0), 0.2])     # log k0, e, u, b, log T, X0
UB = np.array([np.log(0.5), 5.0, 20.0, 3.0, np.log(300.0), 2.0])
PRIOR_W = 0.3

def simulate(theta, d):
    lk0, e, u, b, lT, X0 = theta
    k0, T = np.exp(lk0), np.exp(lT)
    s = d.cofeed_in.values / S_CO; days = d.day.values
    X = np.empty(len(d)); X[0] = X0
    for n in range(1, len(d)):
        xt = max(1.0 + b * s[n], 0.05)
        X[n] = xt + (X[n - 1] - xt) * np.exp(-(days[n] - days[n - 1]) / T)
    k = k0 * X * (1.0 + e * s) / (1.0 + u * s)
    return d.toluene_in.values * np.exp(-k * TAU), X

def fit_adaptive(d):
    d = d.sort_values("day"); yl = np.log(d.toluene_out.values)
    resid = lambda th: np.concatenate([np.log(simulate(th, d)[0]) - yl, PRIOR_W * np.array([th[1], th[2], th[3]])])
    best = None
    for T0 in [5.0, 20.0, 60.0]:
        for X00 in [0.6, 1.0]:
            x0 = np.clip(np.array([np.log(0.05), 0.1, 0.1, 0.1, np.log(T0), X00]), LB + 1e-6, UB - 1e-6)
            sol = least_squares(resid, x0, bounds=(LB, UB))
            if best is None or sol.cost < best.cost:
                best = sol
    return best.x

# ---- in-sample fit (diagnostic)
fits, rows = {}, []
for R in ["A", "B"]:
    d = df[df.reactor == R].sort_values("day")
    th = fit_adaptive(d); c_ad, X = simulate(th, d)
    k0s, kis = fit_mech(d, R); c_st = mech_out(k0s, kis, d.toluene_in.values, d.cofeed_in.values)
    fits[R] = dict(day=d.day.values, obs=d.toluene_out.values, adaptive=c_ad, static=c_st, X=X)
    lk0, e, u, b, lT, X0 = th
    rows.append(dict(reactor=R, k0=round(np.exp(lk0), 4), e=round(e, 2), u=round(u, 2), b=round(b, 2),
                     T_days=round(np.exp(lT), 1), X0=round(X0, 2),
                     MAE_static=round(mae(c_st - d.toluene_out.values)), MAE_adaptive=round(mae(c_ad - d.toluene_out.values))))
tabA5 = pd.DataFrame(rows); print(tabA5.to_string(index=False))
tabA5.to_csv(f"{OUT}/A5_adaptive_insample.csv", index=False)

fig, axes = plt.subplots(2, 2, figsize=(12, 6), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
for j, R in enumerate(["A", "B"]):
    f_ = fits[R]; ax, axx = axes[0, j], axes[1, j]
    for a in (ax, axx):
        for x in [62.5, 107.5, 152.5]:
            a.axvline(x, color=SEP, lw=1, zorder=0)
    ax.plot(f_["day"], f_["obs"], "o", ms=4, color=INK, label="Observed", zorder=3)
    ax.plot(f_["day"], f_["static"], "-", lw=2, color=C1, label="Static mechanistic")
    ax.plot(f_["day"], f_["adaptive"], "-", lw=2, color=C2, label="Adaptive mechanistic")
    ax.set_title(NAME[R], loc="left"); ax.grid(axis="y", color=GRID, lw=0.8)
    axx.plot(f_["day"], f_["X"], "-", lw=2, color=C2); axx.set_xlabel("Day"); axx.grid(axis="y", color=GRID, lw=0.8)
axes[0, 0].set_ylabel("Toluene outlet (mg/m³)"); axes[1, 0].set_ylabel("Activity X(t)")
axes[0, 0].legend(frameon=False, fontsize=9, loc="upper left")
fig.tight_layout(); fig.savefig(f"{OUT}/Fig5_A_adaptive_model.png", dpi=200); plt.show()

# ---- forward forecasting with the adaptive model
rec = []
for d0 in sorted(df.day.unique()):
    if d0 < FIRST_FORECAST_DAY:
        continue
    for R in ["A", "B"]:
        hist = df[(df.reactor == R) & (df.day < d0)].sort_values("day")
        full = df[(df.reactor == R) & (df.day <= d0)].sort_values("day")
        c_full, _ = simulate(fit_adaptive(hist), full)
        ratio = hist.toluene_out.values[-1] / c_full[-2]
        row = full.iloc[-1]
        for name, p in [("Adaptive mechanistic", c_full[-1]), ("Hybrid: adaptive + last residual", c_full[-1] * ratio)]:
            rec.append(dict(reactor=R, day=row.day, phase=row.phase, phase_start=int(row.phase_start),
                            observed=row.toluene_out, model=name, pred=float(p)))
fc6 = pd.DataFrame(rec); fc6["err"] = fc6.pred - fc6.observed
allfc = pd.concat([fc, fc6], ignore_index=True)

ORDER_F = ["Persistence", "Mechanistic", "Direct Ridge (with lag)", "Direct XGBoost (with lag)",
           "Hybrid: mech + last residual", "Hybrid: mech + Ridge residual", "Hybrid: mech + XGBoost residual",
           "Adaptive mechanistic", "Hybrid: adaptive + last residual"]
rows = []
for name in ORDER_F:
    s = allfc[allfc.model == name]
    rows.append(dict(model=name, MAE_all=round(mae(s.err)), MAE_phase_change=round(mae(s[s.phase_start == 1].err)),
                     MAE_within_phase=round(mae(s[s.phase_start == 0].err)), RMSE_all=round(rmse(s.err)),
                     Bias_all=round(float(s.err.mean()))))
tab_f = pd.DataFrame(rows)
tab_f["skill_vs_persistence"] = (1 - tab_f.MAE_all / tab_f.set_index("model").loc["Persistence", "MAE_all"]).round(2)
print(f"\nForecasts per model: {int((allfc.model == 'Persistence').sum())} "
      f"({int(((allfc.model == 'Persistence') & (allfc.phase_start == 1)).sum())} are first samples after a phase change)")
print(tab_f.to_string(index=False))
tab_f.to_csv(f"{OUT}/A5_forecast_results.csv", index=False)

# ---- statistics vs persistence, 95% CI by (reactor, phase) blocks
fa = allfc.copy(); fa["block"] = fa.reactor + "-" + fa.phase
base = fa[fa.model == "Persistence"].set_index(["reactor", "day"]).sort_index()
rows = []
for name in ORDER_F[1:]:
    s = fa[fa.model == name].set_index(["reactor", "day"]).sort_index().loc[base.index]
    d_, lo, hi = block_boot_diff(s.err.values, base.err.values, base.block.values, "mae")
    rows.append(dict(model=name, dMAE_vs_persistence=round(d_), CI95_low=round(lo), CI95_high=round(hi),
                     verdict=verdict(lo, hi, "worse than persistence", "better than persistence")))
stats_f = pd.DataFrame(rows)
print("\n=== Forecast MAE difference vs persistence (positive = worse), 95% CI")
print(stats_f.to_string(index=False))
stats_f.to_csv(f"{OUT}/A5_forecast_stats.csv", index=False)

# ---- Figure 4: forecasts
SHOW = {"Persistence": C3, "Hybrid: mech + last residual": C1, "Hybrid: adaptive + last residual": C2}
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), sharey=True)
for ax, R in zip(axes, ["A", "B"]):
    obs = df[df.reactor == R]
    for x in [62.5, 107.5, 152.5]:
        ax.axvline(x, color=SEP, lw=1, zorder=0)
    for x, ph in zip([40, 85, 130, 175], ["I", "II", "III", "IV"]):
        ax.text(x, 0.97, ph, ha="center", va="top", color=INK2, transform=ax.get_xaxis_transform())
    ax.plot(obs.day, obs.toluene_out, "o", ms=4, color=INK, label="Observed", zorder=3)
    for name, col in SHOW.items():
        s = allfc[(allfc.model == name) & (allfc.reactor == R)].sort_values("day")
        ax.plot(s.day, s.pred, "-", lw=2, color=col, label=name)
    ax.set_title(NAME[R], loc="left", pad=12); ax.set_xlabel("Day"); ax.grid(axis="y", color=GRID, lw=0.8)
axes[0].set_ylabel("Toluene outlet (mg/m³)"); axes[0].set_ylim(-30, 1250)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, frameon=False)
fig.tight_layout(rect=(0, 0.08, 1, 1)); fig.savefig(f"{OUT}/Fig4_A_forecasts.png", dpi=200); plt.show()

## Part B — Dataset B: H₂S biochar biofilter (repeated-measurement leakage)

In [ ]:
# ============ B1. Data, models, physics ============
z = pd.read_excel(find("DataSet.xlsx"))
z.columns = [c.strip() for c in z.columns]
z["cond"] = z.MC.astype(str) + "_" + z.EBRT.astype(str) + "_" + z.H2S_IN.astype(str)
X_Z = ["MC", "EBRT", "H2S_IN"]
yz = z.RE.values
pooled_sd = float(np.sqrt((z.groupby("cond").RE.std(ddof=1) ** 2).mean()))
print(len(z), "rows |", z.cond.nunique(), "conditions |", z.groupby("cond").size().unique(), "repeats each")
print(f"Repeat noise (pooled SD) = {pooled_sd:.2f} RE points")

def mlr():
    return make_pipeline(PolynomialFeatures(2, include_bias=False), StandardScaler(), LinearRegression())
def svr_rbf():
    return TransformedTargetRegressor(regressor=make_pipeline(StandardScaler(),
               SVR(kernel="rbf", C=100, epsilon=0.05, gamma="scale")), transformer=StandardScaler())
def svr_poly3():
    return TransformedTargetRegressor(regressor=make_pipeline(StandardScaler(),
               SVR(kernel="poly", degree=3, coef0=1, C=10, epsilon=0.05)), transformer=StandardScaler())

# physics: plug-flow first-order removal; log k linear in moisture (identifiable with 2 moisture levels)
def phys_pred_z(p, d):
    a, b, e = p
    m = (d.MC.values - 60) / 30
    k = np.exp(a + b * m) * (d.H2S_IN.values / 180.0) ** e
    return 100 * (1 - np.exp(-k * d.EBRT.values))
def phys_fit_z(d):
    return least_squares(lambda p: phys_pred_z(p, d) - d.RE.values, x0=[np.log(0.05), 0, 0]).x

PHYS = "Physics (first-order, k(MC))"
MODELS_Z = ["Train mean", PHYS, "MLR quadratic", "SVR-RBF", "SVR-poly3"]
MAKERS_Z = {"MLR quadratic": mlr, "SVR-RBF": svr_rbf, "SVR-poly3": svr_poly3}

def predict_z(name, tr, te):
    if name == "Train mean":
        return np.full(len(te), tr.RE.mean())
    if name == PHYS:
        return phys_pred_z(phys_fit_z(tr), te)
    m = MAKERS_Z[name](); m.fit(tr[X_Z], tr.RE)
    return m.predict(te[X_Z])

def metr(yt, yp):
    e = yp - yt
    return dict(RMSE=rmse(e), MAE=mae(e), R2=r2(yt, yp))

In [ ]:
# ============ B2. Four ways of testing ============
rows, preds_z = [], {}
ss = ShuffleSplit(n_splits=20, test_size=0.2, random_state=0)
for k in MODELS_Z:
    ms = [metr(yz[te], predict_z(k, z.iloc[tr], z.iloc[te])) for tr, te in ss.split(z)]
    rows.append(dict(scheme="Random 80/20 (as in the paper)", model=k, **pd.DataFrame(ms).mean().to_dict()))
logo = LeaveOneGroupOut()
SCHEMES_Z = {"Leave-one-condition-out": z.cond, "Leave-one-MC-level-out": z.MC, "Leave-one-EBRT-level-out": z.EBRT}
for s, groups in SCHEMES_Z.items():
    p = {k: np.full(len(z), np.nan) for k in MODELS_Z}
    for tr, te in logo.split(z, groups=groups):
        for k in MODELS_Z:
            p[k][te] = predict_z(k, z.iloc[tr], z.iloc[te])
    preds_z[s] = p
    for k in MODELS_Z:
        rows.append(dict(scheme=s, model=k, **metr(yz, p[k])))
res_z = pd.DataFrame(rows)
res_z[["RMSE", "MAE"]] = res_z[["RMSE", "MAE"]].round(2); res_z["R2"] = res_z["R2"].round(3)
for s in res_z.scheme.unique():
    print(f"\n=== {s}   (RE, percentage points; repeat noise = {pooled_sd:.2f})")
    print(res_z[res_z.scheme == s].drop(columns="scheme").to_string(index=False))
res_z.to_csv(f"{OUT}/B2_validation_results.csv", index=False)

for s, col in [("Leave-one-MC-level-out", "MC"), ("Leave-one-EBRT-level-out", "EBRT")]:
    t = pd.DataFrame({k: pd.Series(np.abs(preds_z[s][k] - yz)).groupby(z[col].values).mean().round(2) for k in MODELS_Z})
    print(f"\n=== MAE per held-out {col} level ({s})"); print(t.to_string())
    t.to_csv(f"{OUT}/B2_mae_per_{col}_level.csv")

# ---- statistics vs physics, 95% CI by bootstrapping conditions
rows = []
for s in SCHEMES_Z:
    for k in MODELS_Z:
        if k == PHYS:
            continue
        d_, lo, hi = block_boot_diff(preds_z[s][k] - yz, preds_z[s][PHYS] - yz, z.cond.values, "rmse")
        rows.append(dict(scheme=s, model=k, dRMSE_vs_physics=round(d_, 2), CI95_low=round(lo, 2), CI95_high=round(hi, 2),
                         verdict=verdict(lo, hi, "worse than physics", "better than physics")))
stats_z = pd.DataFrame(rows)
print("\n=== RMSE difference vs physics (positive = worse than physics), 95% CI")
print(stats_z.to_string(index=False))
stats_z.to_csv(f"{OUT}/B2_stats.csv", index=False)

# ---- Figure 6: error per held-out level
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
COLS = [C1, C2, C3, C4, C5]
for ax, (s, col, lab) in zip(axes, [("Leave-one-MC-level-out", "MC", "Held-out moisture content (%)"),
                                    ("Leave-one-EBRT-level-out", "EBRT", "Held-out EBRT (s)")]):
    levels = sorted(z[col].unique()); w = 0.16
    for j, k in enumerate(MODELS_Z):
        vals = [np.mean(np.abs(preds_z[s][k] - yz)[z[col].values == L]) for L in levels]
        ax.bar(np.arange(len(levels)) + (j - 2) * w, vals, width=w * 0.9, color=COLS[j], label=k)
    ax.set_xticks(range(len(levels)))
    ax.set_xticklabels([f"{L}\n({'interpolation' if i == 1 else 'extrapolation'})" for i, L in enumerate(levels)])
    ax.set_xlabel(lab); ax.grid(axis="y", color=GRID, lw=0.8)
axes[0].set_ylabel("MAE of held-out removal efficiency (% points)")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=5, frameon=False, fontsize=9)
fig.tight_layout(rect=(0, 0.1, 1, 1)); fig.savefig(f"{OUT}/Fig6_B_error_by_level.png", dpi=200); plt.show()

## Part C — Both datasets together

In [ ]:
# ============ C1. Inflation ratio: honest error / random-split error ============
ra = res_a.pivot(index="model", columns="scheme", values="MAE").loc[MODELS_A]
infl_a = pd.DataFrame({s: (ra[s] / ra["Random 5-fold (as in literature)"]).round(1) for s in SCHEMES_A})
rz = res_z.pivot(index="model", columns="scheme", values="RMSE").loc[MODELS_Z]
infl_z = pd.DataFrame({s: (rz[s] / rz["Random 80/20 (as in the paper)"]).round(1) for s in SCHEMES_Z})
print("=== Dataset A: honest MAE / random-split MAE"); print(infl_a.to_string())
print("\n=== Dataset B: honest RMSE / random-split RMSE"); print(infl_z.to_string())
infl_a.to_csv(f"{OUT}/C1_inflation_A.csv"); infl_z.to_csv(f"{OUT}/C1_inflation_B.csv")

# ============ Figure 3: same models, random vs honest testing, both datasets ============
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
panels = [
    (axes[0], res_a, MODELS_A, "MAE", ["Random 5-fold (as in literature)", "Leave-one-phase-out", "Cross-reactor"],
     "(a) Dataset A — toluene outlet MAE (mg/m³)", False),
    (axes[1], res_z, MODELS_Z, "RMSE", ["Random 80/20 (as in the paper)", "Leave-one-condition-out", "Leave-one-MC-level-out"],
     "(b) Dataset B — removal efficiency RMSE (% points, log scale)", True),
]
for ax, res_, models, met, schemes, title, logx in panels:
    for j, (s, col) in enumerate(zip(schemes, [C1, C2, C3])):
        sub = res_[res_.scheme == s].set_index("model").loc[models]
        ax.plot(sub[met], np.arange(len(models)) + (j - 1) * 0.22, "o", ms=8, color=col, label=s,
                markeredgecolor="white", markeredgewidth=1.5)
    ax.set_yticks(range(len(models))); ax.set_yticklabels(models); ax.invert_yaxis()
    ax.set_title(title, loc="left", fontsize=10); ax.grid(axis="x", color=GRID, lw=0.8)
    if logx:
        ax.set_xscale("log"); ax.set_xlim(0.4, 20); ax.axvline(pooled_sd, color=INK2, lw=1, ls="--")
        ax.set_xticks([0.5, 1, 2, 5, 10, 20]); ax.set_xticklabels(["0.5", "1", "2", "5", "10", "20"])
        ax.tick_params(axis="x", which="minor", labelbottom=False)
        ax.text(pooled_sd * 1.04, -0.45, "repeat noise", rotation=90, color=INK2, fontsize=8, va="top", ha="left")
    else:
        ax.set_xlim(left=0)
    ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.45, -0.1), ncol=3, fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/Fig3_random_vs_honest_both.png", dpi=200); plt.show()
print("\nAll results saved in:", OUT, "\n", sorted(os.listdir(OUT)))

## Part D — Robustness and transparency checks
D1: nested hyperparameter tuning · D2: forecast start day and horizon · D3: seed variability, drop-one-block robustness and fitted parameters

In [ ]:
# ============ D1. Robustness check: nested hyperparameter tuning ============
# Answers the reviewer question "your ML models were not tuned".
# Outer loop = the SAME validation schemes as A3 and B2.
# Inner loop = hyperparameter search using ONLY the outer training data.
#   Random schemes -> random inner KFold (what the literature does)
#   Grouped schemes -> inner folds grouped the same way (leave-one-group-out)
from sklearn.model_selection import GridSearchCV, GroupKFold

def tuned(est, grid, score):
    return lambda cv: GridSearchCV(est, grid, cv=cv, scoring=score, n_jobs=-1, error_score="raise")

# ---------- Dataset A: search spaces ----------
TUNE_A = {
    "Ridge": tuned(scaled(Ridge()), {"regressor__ridge__alpha": [0.01, 0.1, 1, 10, 100, 1000]},
                   "neg_mean_absolute_error"),
    "SVR-RBF": tuned(scaled(SVR()), {"regressor__svr__C": [0.3, 1, 3, 10, 30, 100],
                                     "regressor__svr__epsilon": [0.01, 0.05, 0.2],
                                     "regressor__svr__gamma": ["scale", 0.03, 0.1, 0.3, 1]},
                     "neg_mean_absolute_error"),
    "Random Forest": tuned(RandomForestRegressor(n_estimators=200, random_state=0),
                           {"min_samples_leaf": [1, 3, 5, 10], "max_features": [1.0, 0.6, 0.3]},
                           "neg_mean_absolute_error"),
    "XGBoost": tuned(XGBRegressor(subsample=0.8, random_state=0, n_jobs=1),
                     {"max_depth": [1, 2, 3, 4], "learning_rate": [0.03, 0.1],
                      "n_estimators": [100, 300], "min_child_weight": [1, 3]},
                     "neg_mean_absolute_error"),
}

def evaluate_tuned_a(folds, inner):
    """inner = 'random' or name of the grouping column used for inner leave-one-group-out."""
    pred, chosen = {k: np.full(len(df), np.nan) for k in TUNE_A}, []
    for tr_idx, te_idx in folds:
        tr, te = df.iloc[tr_idx], df.iloc[te_idx]
        if inner == "random":
            cv, groups = KFold(5, shuffle=True, random_state=0), None
        else:
            groups = tr[inner].values
            cv = LeaveOneGroupOut()
        for name, make in TUNE_A.items():
            gs = make(cv).fit(tr[FEATURES], tr.toluene_out, groups=groups)
            pred[name][te_idx] = gs.predict(te[FEATURES])
            chosen.append(dict(model=name, **{k.split("__")[-1]: v for k, v in gs.best_params_.items()}))
    return pred, chosen

rows, chosen_all, pred_tuned_a = [], [], {}
# random 5-fold x 5 repeats (inner loop random, as in the literature)
rand_t = []
for s in range(5):
    p, ch = evaluate_tuned_a(list(KFold(5, shuffle=True, random_state=s).split(df)), "random")
    rand_t.append(p); chosen_all += [dict(scheme="Random", **c) for c in ch]
for name in TUNE_A:
    ms = pd.DataFrame([metrics_a(r[name]) for r in rand_t]).mean()
    rows.append(dict(dataset="A", scheme="Random 5-fold (as in literature)", model=name, **ms))
# grouped schemes: inner loop grouped by phase (for cross-reactor: phases of the one training reactor)
for scheme, folds in SCHEMES_A.items():
    p, ch = evaluate_tuned_a(folds, "group")
    pred_tuned_a[scheme] = p; chosen_all += [dict(scheme=scheme, **c) for c in ch]
    for name in TUNE_A:
        rows.append(dict(dataset="A", scheme=scheme, model=name, **metrics_a(p[name])))
tun_a = pd.DataFrame(rows)

# ---------- Dataset B: search spaces ----------
TUNE_Z = {
    "MLR quadratic": tuned(make_pipeline(PolynomialFeatures(2, include_bias=False), StandardScaler(), LinearRegression()),
                           {"polynomialfeatures__degree": [1, 2, 3]}, "neg_root_mean_squared_error"),
    "SVR-RBF": tuned(TransformedTargetRegressor(regressor=make_pipeline(StandardScaler(), SVR(kernel="rbf")),
                                                transformer=StandardScaler()),
                     {"regressor__svr__C": [1, 10, 100, 1000], "regressor__svr__epsilon": [0.01, 0.05, 0.2],
                      "regressor__svr__gamma": ["scale", 0.1, 0.3, 1, 3]}, "neg_root_mean_squared_error"),
    "SVR-poly3": tuned(TransformedTargetRegressor(regressor=make_pipeline(StandardScaler(), SVR(kernel="poly")),
                                                  transformer=StandardScaler()),
                       {"regressor__svr__C": [0.1, 1, 10, 100], "regressor__svr__epsilon": [0.01, 0.05, 0.2],
                        "regressor__svr__degree": [2, 3], "regressor__svr__coef0": [0, 1]},
                       "neg_root_mean_squared_error"),
}
INNER_Z = {"Leave-one-condition-out": "cond", "Leave-one-MC-level-out": "MC", "Leave-one-EBRT-level-out": "EBRT"}

def fit_pred_z(name, tr, te, cv, groups):
    gs = TUNE_Z[name](cv).fit(tr[X_Z], tr.RE, groups=groups)
    return gs.predict(te[X_Z]), gs.best_params_

rows = []
for name in TUNE_Z:
    ms = []
    for tr, te in ShuffleSplit(n_splits=20, test_size=0.2, random_state=0).split(z):
        p, bp = fit_pred_z(name, z.iloc[tr], z.iloc[te], KFold(5, shuffle=True, random_state=0), None)
        ms.append(metr(yz[te], p))
        chosen_all.append(dict(scheme="B Random", model=name, **{k.split("__")[-1]: v for k, v in bp.items()}))
    rows.append(dict(dataset="B", scheme="Random 80/20 (as in the paper)", model=name, **pd.DataFrame(ms).mean().to_dict()))
pred_tuned_z = {}
for s, groups in SCHEMES_Z.items():
    p = {k: np.full(len(z), np.nan) for k in TUNE_Z}
    for tr, te in LeaveOneGroupOut().split(z, groups=groups):
        trd = z.iloc[tr]
        g_in = trd[INNER_Z[s]].values
        # condition-out: 17 training conditions -> 5 grouped folds; level-out: 2 training levels -> leave one out
        cv = GroupKFold(5) if s == "Leave-one-condition-out" else LeaveOneGroupOut()
        for name in TUNE_Z:
            p[name][te], bp = fit_pred_z(name, trd, z.iloc[te], cv, g_in)
            chosen_all.append(dict(scheme="B " + s, model=name, **{k.split("__")[-1]: v for k, v in bp.items()}))
    pred_tuned_z[s] = p
    for name in TUNE_Z:
        rows.append(dict(dataset="B", scheme=s, model=name, **metr(yz, p[name])))
tun_z = pd.DataFrame(rows)

# ---------- Compare fixed vs tuned ----------
fixA = res_a.rename(columns={"MAE": "MAE_fixed", "R2": "R2_fixed"})[["scheme", "model", "MAE_fixed", "R2_fixed"]]
cmpA = tun_a.rename(columns={"MAE": "MAE_tuned", "R2": "R2_tuned"})[["scheme", "model", "MAE_tuned", "R2_tuned"]].merge(fixA)
cmpA[["MAE_tuned"]] = cmpA[["MAE_tuned"]].round(0); cmpA["R2_tuned"] = cmpA["R2_tuned"].round(2)
fixZ = res_z.rename(columns={"RMSE": "RMSE_fixed", "R2": "R2_fixed"})[["scheme", "model", "RMSE_fixed", "R2_fixed"]]
cmpZ = tun_z.rename(columns={"RMSE": "RMSE_tuned", "R2": "R2_tuned"})[["scheme", "model", "RMSE_tuned", "R2_tuned"]].merge(fixZ)
cmpZ["RMSE_tuned"] = cmpZ["RMSE_tuned"].round(2); cmpZ["R2_tuned"] = cmpZ["R2_tuned"].round(3)
print("=== Dataset A: fixed vs nested-tuned hyperparameters (toluene outlet MAE, mg/m3)")
print(cmpA.to_string(index=False))
print("\n=== Dataset B: fixed vs nested-tuned hyperparameters (RE RMSE, % points)")
print(cmpZ.to_string(index=False))
cmpA.to_csv(f"{OUT}/D1_A_fixed_vs_tuned.csv", index=False)
cmpZ.to_csv(f"{OUT}/D1_B_fixed_vs_tuned.csv", index=False)

# ---------- Inflation ratios with tuned models ----------
infl = []
for d_, t_, metric, rand_name in [("A", tun_a, "MAE", "Random 5-fold (as in literature)"),
                                  ("B", tun_z, "RMSE", "Random 80/20 (as in the paper)")]:
    for m_ in t_.model.unique():
        base = t_[(t_.scheme == rand_name) & (t_.model == m_)][metric].item()
        for s in t_.scheme.unique():
            if s != rand_name:
                v = t_[(t_.scheme == s) & (t_.model == m_)][metric].item()
                infl.append(dict(dataset=d_, model=m_, scheme=s, inflation_tuned=round(v / base, 1)))
infl = pd.DataFrame(infl)
print("\n=== Inflation ratio with tuned models (grouped error / random-split error)")
print(infl.pivot_table(index=["dataset", "model"], columns="scheme", values="inflation_tuned").to_string())
infl.to_csv(f"{OUT}/D1_inflation_tuned.csv", index=False)

# ---------- Statistics: tuned ML vs mechanistic / physics ----------
rows = []
for s in SCHEMES_A:
    for k in TUNE_A:
        d_, lo, hi = block_boot_diff(pred_tuned_a[s][k] - y, pred_store[s]["Mechanistic"] - y, df.group.values, "mae")
        rows.append(dict(dataset="A", scheme=s, model=k + " (tuned)", diff_vs_mech=round(d_), CI95_low=round(lo),
                         CI95_high=round(hi), verdict=verdict(lo, hi, "worse than mechanistic", "better than mechanistic")))
for s in SCHEMES_Z:
    for k in TUNE_Z:
        d_, lo, hi = block_boot_diff(pred_tuned_z[s][k] - yz, preds_z[s][PHYS] - yz, z.cond.values, "rmse")
        rows.append(dict(dataset="B", scheme=s, model=k + " (tuned)", diff_vs_mech=round(d_, 2), CI95_low=round(lo, 2),
                         CI95_high=round(hi, 2), verdict=verdict(lo, hi, "worse than physics", "better than physics")))
stats_t = pd.DataFrame(rows)
print("\n=== Tuned ML minus mechanistic/physics (A: MAE, B: RMSE; positive = ML worse), 95% CI")
print(stats_t.to_string(index=False))
stats_t.to_csv(f"{OUT}/D1_stats_tuned.csv", index=False)

# ---------- Which hyperparameters were chosen (random vs grouped inner loop) ----------
ch = pd.DataFrame(chosen_all)
ch.to_csv(f"{OUT}/D1_chosen_hyperparameters.csv", index=False)
print("\n=== Most frequently chosen setting per model and scheme")
pcols = [c for c in ch.columns if c not in ("scheme", "model")]
summ = (ch.fillna("-").astype(str).groupby(["scheme", "model"])[pcols]
          .agg(lambda s: s.value_counts().index[0]).reset_index())
print(summ.to_string(index=False))


In [ ]:
# ============ D2. Robustness check: forecast start day and horizon ============
# Answers the reviewer question "does 'nothing beats persistence' depend on starting at day 40
# and forecasting only 5 days ahead?"
# Target day d0, forecast origin o = d0 - 5*h. Only data with day <= o are used (strictly forward).
# Inlet concentrations at d0 are treated as known (planned operating schedule), as in A4/A5.
import itertools

START_DAYS = [40, 50, 60, 80, 100]     # first target day
HORIZONS = [1, 2, 3]                   # steps of 5 days -> 5, 10, 15 days ahead
MIN_LAG_DAY = 25                       # the ML models need >= 2 (lag, target) pairs per reactor before the origin
MODELS_S = ["Persistence", "Mechanistic", "Hybrid: mech + last residual", "Hybrid: adaptive + last residual",
            "Direct Ridge (with lag)", "Direct XGBoost (with lag)"]
LAG_FEATS = ["y_lag", "toluene_in", "cofeed_in", "cofeed_lag", "phase_changed", "day_in_phase", "is_EA", "is_hexane"]

def lagged(h):
    d = df.copy(); g_ = d.groupby("reactor")
    d["y_lag"] = g_.toluene_out.shift(h)
    d["cofeed_lag"] = g_.cofeed_in.shift(h)
    d["phase_changed"] = (d.phase != g_.phase.shift(h)).astype(int)
    return d

_mech_cache, _adapt_cache, _ml_cache = {}, {}, {}

def mech_params(o):
    if o not in _mech_cache:
        tr = df[df.day <= o]
        _mech_cache[o] = {r_: fit_mech(tr, r_) for r_ in ["A", "B"]}
    return _mech_cache[o]

def adapt_theta(R, o):
    if (R, o) not in _adapt_cache:
        _adapt_cache[(R, o)] = fit_adaptive(df[(df.reactor == R) & (df.day <= o)])
    return _adapt_cache[(R, o)]

def ml_models(h, o, dh):
    if (h, o) not in _ml_cache:
        trd = dh[(dh.day <= o)].dropna(subset=["y_lag"])
        _ml_cache[(h, o)] = {"Direct Ridge (with lag)": ridge().fit(trd[LAG_FEATS], trd.toluene_out),
                             "Direct XGBoost (with lag)": xgb().fit(trd[LAG_FEATS], trd.toluene_out)}
    return _ml_cache[(h, o)]

rec = []
for h in HORIZONS:
    dh = lagged(h)
    for d0 in sorted(df.day.unique()):
        o = d0 - 5 * h
        if o - 5 * h < MIN_LAG_DAY:
            continue
        te = dh[dh.day == d0].copy()
        prm = mech_params(o)
        te["C_mech"] = add_mech(te, prm)
        mls = ml_models(h, o, dh)
        for _, row in te.iterrows():
            R = row.reactor
            at_o = df[(df.reactor == R) & (df.day == o)].iloc[0]
            c_mech_o = mech_out(*prm[R], at_o.toluene_in, at_o.cofeed_in)
            full = df[(df.reactor == R) & (df.day <= d0)].sort_values("day")
            c_ad = simulate(adapt_theta(R, o), full)[0]
            c_ad_o = c_ad[full.day.values == o][0]
            x = row[LAG_FEATS].to_frame().T.astype(float)
            p = {"Persistence": at_o.toluene_out,
                 "Mechanistic": row.C_mech,
                 "Hybrid: mech + last residual": row.C_mech + (at_o.toluene_out - c_mech_o),
                 "Hybrid: adaptive + last residual": c_ad[-1] * at_o.toluene_out / c_ad_o}
            for k, m in mls.items():
                p[k] = float(m.predict(x)[0])
            for k, v in p.items():
                rec.append(dict(h=h, d0=d0, reactor=R, phase=row.phase, block=R + "-" + row.phase,
                                phase_change=int(row.phase_changed), observed=row.toluene_out, model=k, pred=float(v)))
sens = pd.DataFrame(rec); sens["err"] = sens.pred - sens.observed

rows, srows = [], []
for S, h in itertools.product(START_DAYS, HORIZONS):
    if S - 10 * h < MIN_LAG_DAY:            # same target days for every model; skip infeasible settings
        continue
    s = sens[(sens.h == h) & (sens.d0 >= S)]
    base = s[s.model == "Persistence"].sort_values(["reactor", "d0"])
    for k in MODELS_S:
        sk = s[s.model == k].sort_values(["reactor", "d0"])
        rows.append(dict(start_day=S, horizon_days=5 * h, n=len(sk), model=k, MAE=round(mae(sk.err)),
                         MAE_phase_change=round(mae(sk[sk.phase_change == 1].err)),
                         MAE_within_phase=round(mae(sk[sk.phase_change == 0].err))))
        if k != "Persistence":
            d_, lo, hi = block_boot_diff(sk.err.values, base.err.values, base.block.values, "mae")
            srows.append(dict(start_day=S, horizon_days=5 * h, model=k, dMAE_vs_persistence=round(d_),
                              CI95_low=round(lo), CI95_high=round(hi),
                              verdict=verdict(lo, hi, "worse than persistence", "better than persistence")))
tab_s = pd.DataFrame(rows); stats_s = pd.DataFrame(srows)

print("=== Check: start day 40, 5 days ahead should reproduce A4/A5 (Persistence 78, mech + last residual 87 ...)")
print(tab_s[(tab_s.start_day == 40) & (tab_s.horizon_days == 5)].to_string(index=False))
print("\n=== Forecast MAE (mg/m3) by start day and horizon")
print(tab_s.pivot_table(index="model", columns=["start_day", "horizon_days"], values="MAE").loc[MODELS_S].to_string())
print("\n=== Best model per setting")
best = tab_s.loc[tab_s.groupby(["start_day", "horizon_days"]).MAE.idxmin(), ["start_day", "horizon_days", "model", "MAE"]]
print(best.to_string(index=False))
print("\n=== MAE difference vs persistence (positive = worse), 95% CI by (reactor, phase) blocks")
print(stats_s.to_string(index=False))
print("\n=== Verdict summary (count of settings)")
print(pd.crosstab(stats_s.model, stats_s.verdict).to_string())
tab_s.to_csv(f"{OUT}/D2_forecast_sensitivity.csv", index=False)
stats_s.to_csv(f"{OUT}/D2_forecast_sensitivity_stats.csv", index=False)

# ---- Supplementary figure: MAE vs horizon for two start days
SHOW_S = {"Persistence": C3, "Hybrid: mech + last residual": C1, "Hybrid: adaptive + last residual": C2,
          "Direct XGBoost (with lag)": C4, "Mechanistic": INK2}
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, S in zip(axes, [60, 100]):
    t = tab_s[tab_s.start_day == S]
    for k, col in SHOW_S.items():
        tk = t[t.model == k].sort_values("horizon_days")
        ax.plot(tk.horizon_days, tk.MAE, "-o", lw=2, ms=5, color=col, label=k)
    ax.set_title(f"Forecasts from day {S}", loc="left"); ax.set_xlabel("Forecast horizon (days)")
    ax.set_xticks([5, 10, 15]); ax.grid(axis="y", color=GRID, lw=0.8)
axes[0].set_ylabel("MAE, toluene outlet (mg/m³)")
h_, l_ = axes[0].get_legend_handles_labels()
fig.legend(h_, l_, loc="lower center", ncol=3, frameon=False)
fig.tight_layout(rect=(0, 0.13, 1, 1)); fig.savefig(f"{OUT}/FigS1_forecast_sensitivity.png", dpi=200); plt.show()


In [ ]:
# ============ D3. Transparency: seed variability, drop-one-block robustness, fitted parameters ============
# (uses objects created in A2, A3, A5, B1, B2)

# ---- 1. Variability of the random-split results across seeds / repeats
rows = []
for s_, r in enumerate(rand):                                   # Dataset A: 5 seeds of random 5-fold
    for name in MODELS_A:
        rows.append(dict(dataset="A", repeat=s_, model=name, **metrics_a(r[name])))
for i, (tr, te) in enumerate(ShuffleSplit(n_splits=20, test_size=0.2, random_state=0).split(z)):   # Dataset B: 20 splits
    for k in MODELS_Z:
        rows.append(dict(dataset="B", repeat=i, model=k, **metr(yz[te], predict_z(k, z.iloc[tr], z.iloc[te]))))
seeds = pd.DataFrame(rows)
seeds["err"] = np.where(seeds.dataset == "A", seeds.MAE, seeds.RMSE)       # primary metric per dataset
seed_sum = (seeds.groupby(["dataset", "model"], sort=False)
            .agg(err_mean=("err", "mean"), err_sd=("err", "std"), err_min=("err", "min"), err_max=("err", "max"),
                 R2_min=("R2", "min"), R2_max=("R2", "max")).round(2).reset_index())
print("=== Random-split variability across repeats (A: MAE mg/m3, 5 seeds; B: RMSE % points, 20 splits)")
print(seed_sum.to_string(index=False))
seed_sum.to_csv(f"{OUT}/D3_random_split_variability.csv", index=False)

# inflation ratio computed against every individual random repeat -> range
rows = []
for d_, honest, met in [("A", res_a, "MAE"), ("B", res_z, "RMSE")]:
    rnd_name = "Random 5-fold (as in literature)" if d_ == "A" else "Random 80/20 (as in the paper)"
    for s in honest.scheme.unique():
        if s == rnd_name:
            continue
        for m_ in honest.model.unique():
            v = honest[(honest.scheme == s) & (honest.model == m_)][met].item()
            r_ = seeds[(seeds.dataset == d_) & (seeds.model == m_)].err
            rows.append(dict(dataset=d_, scheme=s, model=m_, inflation_mean=round(v / r_.mean(), 1),
                             inflation_min=round(v / r_.max(), 1), inflation_max=round(v / r_.min(), 1)))
infl_rng = pd.DataFrame(rows)
print("\n=== Inflation ratio (grouped error / random-split error): value vs mean, and range over individual repeats")
print(infl_rng.to_string(index=False))
infl_rng.to_csv(f"{OUT}/D3_inflation_range.csv", index=False)

# ---- 2. Dataset A, leave-one-phase-out: do the conclusions survive when one block is removed?
lp = pred_store["Leave-one-phase-out"]
rows = []
for g_ in sorted(df.group.unique()):
    keep = (df.group != g_).values
    maes = {k: round(metrics_a(v, keep)["MAE"]) for k, v in lp.items()}
    best = min(maes, key=maes.get)
    rows.append(dict(dropped_block=g_, **maes, best_model=best,
                     any_ML_below_mean=any(maes[k] < maes["Train mean"] for k in ML_A)))
drop1 = pd.DataFrame(rows)
print("\n=== Leave-one-phase-out MAE recomputed with one (reactor, phase) block removed")
print(drop1.to_string(index=False))
drop1.to_csv(f"{OUT}/D3_drop_one_block_A.csv", index=False)

# ---- 3. Fitted mechanistic parameters (all data) and their range across folds
rows = []
for R in ["A", "B"]:
    k0, ki = fit_mech(df[df.reactor == R], R)
    rows.append(dict(model="Inhibition (Dataset A)", fit="all data, reactor " + R, k0_per_s=round(k0, 4), Ki_mg_m3=round(ki)))
for g_ in sorted(df.group.unique()):                    # fold-wise values under leave-one-phase-out
    tr = df[df.group != g_]; R = g_.split("-")[0]
    k0, ki = fit_mech(tr, R)
    rows.append(dict(model="Inhibition (Dataset A)", fit=f"LOPO fold without {g_}", k0_per_s=round(k0, 4),
                     Ki_mg_m3=(round(ki) if np.isfinite(ki) else "inf")))
pa, pb, pe = phys_fit_z(z)
rows.append(dict(model="First-order k(MC) (Dataset B)", fit="all data", a=round(pa, 3), b=round(pb, 3), e=round(pe, 3),
                 k_at_60pct_180ppm_per_s=round(np.exp(pa), 4)))
for s, col in [("MC level out", "MC"), ("EBRT level out", "EBRT")]:
    for lvl in sorted(z[col].unique()):
        a_, b_, e_ = phys_fit_z(z[z[col] != lvl])
        rows.append(dict(model="First-order k(MC) (Dataset B)", fit=f"{s}: without {col}={lvl}",
                         a=round(a_, 3), b=round(b_, 3), e=round(e_, 3)))
params = pd.DataFrame(rows)
print("\n=== Fitted mechanistic parameters")
print(params.fillna("").to_string(index=False))
params.to_csv(f"{OUT}/D3_mechanistic_parameters.csv", index=False)
print("\nAdaptive model parameters (A5):"); print(tabA5.to_string(index=False))


In [ ]:
# ============ Download all results as one zip ============
import shutil
base = os.path.dirname(os.path.abspath(OUT))
shutil.make_archive(os.path.join(base, "paper_figures_and_tables"), "zip", OUT)
print("Saved:", os.path.join(base, "paper_figures_and_tables.zip"))
print(sorted(os.listdir(OUT)))
try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(os.path.join(base, "paper_figures_and_tables.zip"))))
except Exception:
    pass
